# Mixture-of-Experts: build it, route it, watch the collapse — and the fix

Imagine a hospital with one brilliant generalist who sees *every* patient — the broken arm, the rare disease, the routine checkup — keeping all of medicine fresh for each one. That is a **dense** network: every parameter fires for every input. A **Mixture-of-Experts (MoE)** hospital instead has a *front desk* (the **router**) that glances at each patient and sends them to the two **specialists** (the **experts**) who can actually help; the other ninety-eight stay in their offices. The hospital's *total knowledge* is enormous, but the *work per patient* is tiny. That single reframing — **grow the knowledge, not the work per input** — is why frontier LLMs got cheap enough to scale.

This notebook builds one MoE layer from scratch and proves the two things that matter:

1. **Routing works** — each token is sent to its top-*k* experts and the outputs are combined by the router's gate weights (we print the shapes and *which expert each token went to*).
2. **Load balancing is everything** — left alone, a router **collapses** (a few experts hog every token, the rest go dead). The **auxiliary load-balancing loss** $\mathcal{L}_{\text{aux}} = N\sum_i f_i P_i$ is what prevents it, and we *measure* the difference.

Everything runs on CPU in a few seconds. The companion script is [`mixture_of_experts.py`](mixture_of_experts.py); the full explanation is the [concept page](../07-Mixture-of-Experts.md).

## The problem: dense models pay for every parameter on every token

A transformer block is **attention** (tokens exchange information) plus a **feed-forward network (FFN)** (each token is transformed independently). The FFN — the $d \to 4d \to d$ expansion — holds roughly **two-thirds** of the block's parameters, and in a **dense** model *that entire FFN runs for every token*: the word "the", a closing bracket, a rare technical term, all push through all the weights. That is the waste — because in a dense model **parameters and FLOPs are the same thing** (to use a parameter is to multiply by it), you cannot spend less compute on an easy token and more on a hard one.

MoE breaks that link: replace the one FFN with **$N$ expert FFNs + a router**, and run only the **top-$k$** experts per token. Grow $N$ (capacity) without growing $k$ (compute). The rest of this notebook is that idea, in code.

> **Step 1 — set up the layer.** Pick the dimensions. We keep them tiny so every print is readable: a `d_model=32` token width, `d_ff=64` expert FFNs, `N=8` experts, and top-`k=2` routing. Everything is device-agnostic (CUDA → MPS → CPU) and seeded for reproducibility.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

detected = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
# Pin to CPU so the seeded numbers below are bit-reproducible across machines; the printed
# line is honest about it (we never print one device and compute on another).
device = "cpu"
torch.manual_seed(0)

D_MODEL, D_FF, N_EXPERTS, TOP_K = 32, 64, 8, 2
line = device if detected == device else f"{device} (detected {detected}; pinned to CPU for reproducibility)"
print("device:", line)
print("torch:", torch.__version__)
print(f"d_model={D_MODEL}, d_ff={D_FF}, N_experts={N_EXPERTS}, top_k={TOP_K}")
print(f"ideal balanced top-1 usage = {100/N_EXPERTS:.1f}% per expert")

device: cpu (detected mps; pinned to CPU for reproducibility)
torch: 2.12.0
d_model=32, d_ff=64, N_experts=8, top_k=2
ideal balanced top-1 usage = 12.5% per expert


> **Step 2 — the MoE layer itself.** A `MoELayer` is just **a router** (one linear map `d_model → n_experts`) plus **N ordinary FFNs**. "Expert" is a grand word for an MLP — there's nothing special inside one; the intelligence is in the *router* deciding who runs. `forward` returns the routed output, the auxiliary load-balancing loss, and each token's top-1 expert (for measuring utilisation later).

In [2]:
class MoELayer(nn.Module):
    def __init__(self, d_model=D_MODEL, d_ff=D_FF, n_experts=N_EXPERTS, k=TOP_K):
        super().__init__()
        self.n_experts, self.k = n_experts, k
        self.router = nn.Linear(d_model, n_experts, bias=False)   # the gating net
        self.experts = nn.ModuleList(                             # N ordinary FFNs
            nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
            for _ in range(n_experts))

    def forward(self, x):                              # x: (tokens, d_model)
        logits = self.router(x)                        # (T, N)  router scores
        probs  = F.softmax(logits, dim=-1)             # (T, N)  g(x), sums to 1 per token
        gate, idx = probs.topk(self.k, dim=-1)         # top-k experts per token
        gate = gate / gate.sum(-1, keepdim=True)       # renormalise the k gates -> convex combine

        y = torch.zeros_like(x)
        for slot in range(self.k):                     # combine the k chosen experts
            for e in range(self.n_experts):
                m = idx[:, slot] == e                  # tokens whose slot-th pick is expert e
                if m.any():
                    y[m] += gate[m, slot:slot+1] * self.experts[e](x[m])

        # load-balancing auxiliary loss (Switch):  L = N * sum_i f_i * P_i
        top1 = idx[:, 0]
        f = F.one_hot(top1, self.n_experts).float().mean(0)   # fraction routed (top-1), HARD count
        P = probs.mean(0)                                     # mean router prob, SOFT/differentiable
        aux = self.n_experts * (f * P).sum()
        return y, aux, top1

print(MoELayer())

MoELayer(
  (router): Linear(in_features=32, out_features=8, bias=False)
  (experts): ModuleList(
    (0-7): 8 x Sequential(
      (0): Linear(in_features=32, out_features=64, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=64, out_features=32, bias=True)
    )
  )
)


## Route one batch: shapes and per-token assignments

> **Step 3 — push 6 tokens through and watch where they go.** This is the whole forward pass made visible: the router produces a score per expert `(T, N)`, a softmax turns scores into probabilities, top-$k$ picks the best $k$ experts per token, and the output has the **same shape as the input** — an MoE layer is a drop-in replacement for an FFN. The printed list shows each token's two chosen experts and their renormalised gate weights. Notice routing is **per-token** — six tokens scatter across many experts, exactly as the page warns (MoE does *not* route whole prompts to a "coding expert").

In [3]:
torch.manual_seed(0)
moe = MoELayer().to(device)                       # place router + experts on the run device
n_tokens = 6
x = torch.randn(n_tokens, D_MODEL, device=device)

logits = moe.router(x)
probs  = F.softmax(logits, dim=-1)
gate, idx = probs.topk(moe.k, dim=-1)
gate = gate / gate.sum(-1, keepdim=True)
y, aux, _ = moe(x)

print(f"input x            : {tuple(x.shape)}   (T={n_tokens} tokens, d_model={D_MODEL})")
print(f"router logits      : {tuple(logits.shape)}   (one score per expert)")
print(f"router probs g(x)  : {tuple(probs.shape)}   (softmax, each row sums to 1)")
print(f"top-{moe.k} experts/token: {tuple(idx.shape)}   (the chosen expert indices)")
print(f"output y           : {tuple(y.shape)}   (same shape as input -- a drop-in FFN)\n")
print(f"each token -> its top-{moe.k} experts (renormalised gate weights):")
for t in range(n_tokens):
    picks = ", ".join(f"E{idx[t, s].item()} (g={gate[t, s].item():.2f})" for s in range(moe.k))
    print(f"  token {t}: {picks}")
print(f"\naux load-balancing loss on this batch: {aux.item():.4f}  (min possible = 1.0)")

input x            : (6, 32)   (T=6 tokens, d_model=32)
router logits      : (6, 8)   (one score per expert)
router probs g(x)  : (6, 8)   (softmax, each row sums to 1)
top-2 experts/token: (6, 2)   (the chosen expert indices)
output y           : (6, 32)   (same shape as input -- a drop-in FFN)

each token -> its top-2 experts (renormalised gate weights):
  token 0: E2 (g=0.55), E6 (g=0.45)
  token 1: E0 (g=0.50), E5 (g=0.50)
  token 2: E7 (g=0.52), E5 (g=0.48)
  token 3: E4 (g=0.75), E5 (g=0.25)
  token 4: E2 (g=0.62), E1 (g=0.38)
  token 5: E1 (g=0.53), E7 (g=0.47)

aux load-balancing loss on this batch: 1.0586  (min possible = 1.0)


## The auxiliary loss, by hand: balanced vs collapsed

> **Step 4 — compute $\mathcal{L}_{\text{aux}} = N\sum_i f_i P_i$ on two routings.** Before training anything, see *why* the penalty works. With $N=4$ experts: a **balanced** routing ($f=P=0.25$ each) bottoms out at exactly **1.000** — the minimum — because a uniform distribution minimises $\sum_i P_i^2$. A **collapsed** routing (one expert grabs 70% of tokens) scores **1.947**, nearly double. Because the penalty rises with $f_i P_i$, its gradient pushes hardest on exactly the over-used experts' probabilities. This is the worked example from the concept page, reproduced numerically.

In [4]:
def aux_loss(f, p, n_experts):
    """L = N * sum_i f_i * P_i for given fraction f and mean-prob P vectors."""
    return float(n_experts * (f * p).sum())

n = 4
f_bal = torch.tensor([0.25, 0.25, 0.25, 0.25], device=device)
p_bal = torch.tensor([0.25, 0.25, 0.25, 0.25], device=device)
# p_col is an ILLUSTRATIVE mean-router-probability vector, skewed the same way as f_col
# to mirror the collapse -- not measured from a run; it shows how the penalty rises.
f_col = torch.tensor([0.70, 0.20, 0.07, 0.03], device=device)
p_col = torch.tensor([0.62, 0.22, 0.10, 0.06], device=device)

l_bal = aux_loss(f_bal, p_bal, n)
l_col = aux_loss(f_col, p_col, n)
print(f"aux loss, BALANCED  routing (f=P=0.25 each) : {l_bal:.3f}   <- the minimum, 1.0")
print(f"aux loss, COLLAPSED routing (one expert 70%): {l_col:.3f}   <- nearly double")
assert l_col > l_bal, "collapsed routing must score a higher aux loss than balanced"
assert abs(l_bal - 1.0) < 1e-6, "perfectly balanced routing must bottom out at exactly 1.0"
print("assert passed: collapsed aux loss > balanced aux loss, and balanced == 1.000")

aux loss, BALANCED  routing (f=P=0.25 each) : 1.000   <- the minimum, 1.0
aux loss, COLLAPSED routing (one expert 70%): 1.947   <- nearly double
assert passed: collapsed aux loss > balanced aux loss, and balanced == 1.000


## The main event: router collapse, and the loss that fixes it

Nothing in the forward pass *requires* the router to use all the experts — and gradient descent actively prefers not to. Early on, a few experts get slightly more tokens by chance → more updates → they get *better* → the router (minimising the same loss) sends them *even more* tokens. This **rich-get-richer** loop's fixed point is **collapse**: a handful of experts do everything while the rest go dead. The only counter-pressure is the auxiliary loss.

> **Step 5 — a training loop that makes the collapse explicit.** We train the layer with **top-1** routing (where collapse is sharpest). Each step we find, per token, the expert that *currently* fits best and push the router toward it — the rich-get-richer driver, made into an explicit signal. Toggling `use_aux` isolates exactly what the aux loss buys.

In [5]:
def train_moe(use_aux, steps=1500, seed=0):
    torch.manual_seed(seed)
    moe = MoELayer(k=1).to(device)                    # top-1: rich-get-richer is sharpest
    opt = torch.optim.Adam(moe.parameters(), lr=3e-3)
    W = torch.randn(D_MODEL, D_MODEL, device=device)  # a fixed nonlinear target map
    for _ in range(steps):
        x = torch.randn(256, D_MODEL, device=device)
        target = torch.tanh(x @ W)
        y, aux, _ = moe(x)
        # rich-get-richer driver: push the router toward each token's currently-best expert
        with torch.no_grad():
            err = torch.stack([((moe.experts[e](x) - target)**2).mean(-1)
                               for e in range(moe.n_experts)], -1)
            want = (-err).argmax(-1)
        loss = F.mse_loss(y, target) + 3.0 * F.cross_entropy(moe.router(x), want)
        if use_aux:
            loss = loss + 4.0 * aux                     # the only thing fighting the collapse
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():                              # measure utilisation on fresh tokens
        x = torch.randn(4096, D_MODEL, device=device); _, _, top1 = moe(x)
        util = torch.bincount(top1, minlength=moe.n_experts).float()
        return util / util.sum() * 100

print("trained two layers (no-aux and with-aux); measuring utilisation next...")

trained two layers (no-aux and with-aux); measuring utilisation next...


> **Step 6 — measure utilisation both ways.** Run the training twice and print each expert's share of tokens. The ideal is **12.5%** each ($1/8$) with **0 dead** experts. Read the two lines side by side: **without** the aux loss the router collapses (a couple of experts hog ~26% each, two go completely dead); **with** it, usage spreads near the ideal and every expert stays alive. That one added term — $N\sum_i f_i P_i$, a handful of lines — is the difference between an MoE that uses its capacity and one that wastes a quarter of it.

In [6]:
print(f"expert utilisation after training (ideal = {100/N_EXPERTS:.1f}% each, 0 dead):\n")
dead = {}
peak = {}
for use_aux in (False, True):
    util = train_moe(use_aux=use_aux)
    dead[use_aux] = int((util < 0.5).sum())
    peak[use_aux] = float(util.max())
    tag = "WITH aux " if use_aux else "NO aux   "
    bars = ", ".join(f"{v:4.1f}" for v in util)
    print(f"  {tag}| util% = [{bars}] | max {peak[use_aux]:.1f}%  dead {dead[use_aux]}/{len(util)}")

# Precondition: the no-aux run must ACTUALLY collapse, or the demo proves nothing.
assert dead[False] >= 1, "no-aux run must show collapse (>=1 dead) for the demo to be meaningful"
# The aux loss must revive dead experts AND flatten the busiest one's share.
assert dead[True] < dead[False], "the aux loss must reduce the number of dead experts"
assert peak[True] < peak[False], "the aux loss must lower the busiest expert's share"
print(f"\nassert passed: collapse is real ({dead[False]} dead no-aux), the aux loss revives it "
      f"({dead[False]} -> {dead[True]} dead), and the peak share drops "
      f"({peak[False]:.1f}% -> {peak[True]:.1f}%).")

expert utilisation after training (ideal = 12.5% each, 0 dead):



  NO aux   | util% = [26.1, 16.8, 12.5,  6.0, 25.7,  0.1, 12.7,  0.0] | max 26.1%  dead 2/8


  WITH aux | util% = [13.5, 14.4, 12.0,  8.6, 16.6,  8.3, 13.4, 13.2] | max 16.6%  dead 0/8

assert passed: collapse is real (2 dead no-aux), the aux loss revives it (2 -> 0 dead), and the peak share drops (26.1% -> 16.6%).


## What we saw

- **An MoE layer is a router + N ordinary FFNs.** The output has the same shape as the input — it drops in where a dense FFN was. The router does one cheap matmul + softmax; only **top-$k$** experts run per token, so per-token compute scales with $k$, not $N$.
- **Routing is per-token.** Six tokens scattered across many experts; a single token's two experts are combined by the router's renormalised gate weights. MoE never routes whole prompts to a "topic expert".
- **The auxiliary loss $\mathcal{L}_{\text{aux}} = N\sum_i f_i P_i$ bottoms out at 1.0 for perfect balance** and rises toward $N$ as routing concentrates — and its gradient flows through the soft $P_i$ to push over-used experts down.
- **Without it, the router collapses** — two experts went dead and a couple hogged ~26% each. **With it**, usage spread near the 12.5% ideal and every expert stayed alive. That is the single most important thing to understand about *training* an MoE.

Next: the active-vs-total parameter arithmetic (why "8×7B" is ~47B total / ~13B active), capacity factors and token dropping, and the routing variants (Switch top-1, Mixtral top-2, DeepSeek fine-grained + shared) — all on the [concept page](../07-Mixture-of-Experts.md).